# Chapter 20 — Sending Email, Texts, and Push Notifications

Source: [Automate the Boring Stuff with Python, 3rd Ed. — Chapter 20](https://automatetheboringstuff.com/3e/chapter20.html)

**Setup note:** Needs real credentials (an app password, or a service like Twilio) to actually send anything — shown as reference code you'd adapt, not run as-is.

This notebook favors *reading* over *writing*: each section has a short explanation followed by real, runnable code. Read it, run it, tweak a value and re-run it — that's the fastest way to build reading fluency.

---


## 20.1 Sending email with smtplib

`smtplib.SMTP_SSL(server, port)` connects to an email provider's outgoing mail server; `.login()` authenticates (use an app password, never your real account password in code); `.sendmail()` sends the message.


In [1]:
# import smtplib
# smtp_obj = smtplib.SMTP_SSL('smtp.gmail.com', 465)
# smtp_obj.login('my_email@gmail.com', 'my_app_password')
# smtp_obj.sendmail('my_email@gmail.com', 'friend@example.com',
#                    'Subject: Hi\n\nThis is the email body.')
# smtp_obj.quit()
print('Core pattern: connect -> login -> sendmail -> quit. Credentials belong in env vars, never hardcoded.')


Core pattern: connect -> login -> sendmail -> quit. Credentials belong in env vars, never hardcoded.


**2026 update — what the book's own worked example actually uses:** `smtplib` above is the general, provider-agnostic way any Python program sends email, and it's still entirely correct. But Gmail specifically requires either an app password or a full OAuth dance to authenticate, which is fiddly to set up by hand. The book's own 3rd-edition example sidesteps this entirely by using the third-party `EZGmail` package for **both** sending and reading Gmail — see 20.2 below, which replaces the older `imapclient`/`pyzmail` approach this notebook previously described (those still work as a general IMAP technique for non-Gmail providers, but they're no longer what this book's current edition teaches).

## 20.2 Sending and reading Gmail with EZGmail

`EZGmail` wraps Gmail's official API behind a much simpler interface than working with `imapclient`/`pyzmail` or raw OAuth. One-time setup: download a `credentials.json` OAuth file from the Google Cloud Console for your account, then call `ezgmail.init()` once — it opens a browser for you to approve access and saves a `token.json` so future runs skip that step. After that: `ezgmail.send(recipient, subject, body)` sends mail, and `ezgmail.search('subject:hello')` returns matching `GmailThread` objects (each with `.messages`, and each message has `.subject`, `.body`, `.sender`, `.attachments`) — no manual IMAP fetch-and-parse needed.

**Environment note:** this needs a real Google Cloud OAuth credentials file and a one-time interactive browser login — neither is available here, so the cell below is illustrative reference code, not executed.

In [2]:
ezgmail_demo_code = '''
import ezgmail

ezgmail.init()   # first run: opens a browser for OAuth consent; later runs reuse the saved token

# Sending
ezgmail.send('friend@example.com', 'Subject line', 'The email body text.')

# Reading / searching
threads = ezgmail.search('subject:hello')
for thread in threads:
    for message in thread.messages:
        print(message.sender, '-', message.subject)
        print(message.body[:200])
        for attachment_name in message.attachments:
            message.downloadAttachment(attachment_name)
'''

print('Needs a real Google Cloud OAuth credentials.json + one-time browser login -- illustrative only.')
print(ezgmail_demo_code)


Needs a real Google Cloud OAuth credentials.json + one-time browser login -- illustrative only.

import ezgmail

ezgmail.init()   # first run: opens a browser for OAuth consent; later runs reuse the saved token

# Sending
ezgmail.send('friend@example.com', 'Subject line', 'The email body text.')

# Reading / searching
threads = ezgmail.search('subject:hello')
for thread in threads:
    for message in thread.messages:
        print(message.sender, '-', message.subject)
        print(message.body[:200])
        for attachment_name in message.attachments:
            message.downloadAttachment(attachment_name)



## 20.3 Texts and push notifications: Twilio (SMS) and ntfy (push)

Sending an actual SMS still means a paid third-party API like **Twilio**: `requests.post()` to Twilio's Messages endpoint with your account SID/auth token as HTTP basic auth. For simple **push notifications** (no phone number needed), the book uses the free **ntfy** service instead: you `POST` a message to `https://ntfy.sh/<your-topic-name>` (a topic is just a name you make up — anyone subscribed to that exact topic via the ntfy app or a browser gets the notification instantly), no account or API key required for the public server.

In [3]:
import requests

# ntfy: a real, working push notification -- no account or API key needed for the public server
topic = 'atbs-notebook-demo-8f2k1x'   # anyone subscribed to this exact topic name would get this
try:
    response = requests.post(f'https://ntfy.sh/{topic}', data='Hello from the ATBS notebook!', timeout=5)
    print('ntfy response status:', response.status_code)
except requests.exceptions.RequestException as e:
    print(f'Could not reach ntfy.sh from this sandbox ({type(e).__name__}) -- network restricted here.')
    print('This exact code works unmodified on a normal internet connection (e.g. in Cursor).')

# Twilio (SMS): needs a real paid account, shown as reference code
twilio_demo_code = '''
import requests
requests.post(
    'https://api.twilio.com/2010-04-01/Accounts/ACCOUNT_SID/Messages.json',
    auth=('ACCOUNT_SID', 'AUTH_TOKEN'),
    data={'From': '+15551234567', 'To': '+15557654321', 'Body': 'Hi from Python!'}
)
'''
print('\nTwilio SMS needs a paid account -- reference code:')
print(twilio_demo_code)


Could not reach ntfy.sh from this sandbox (ProxyError) -- network restricted here.
This exact code works unmodified on a normal internet connection (e.g. in Cursor).

Twilio SMS needs a paid account -- reference code:

import requests
requests.post(
    'https://api.twilio.com/2010-04-01/Accounts/ACCOUNT_SID/Messages.json',
    auth=('ACCOUNT_SID', 'AUTH_TOKEN'),
    data={'From': '+15551234567', 'To': '+15557654321', 'Body': 'Hi from Python!'}
)

